# Site data builder — rating v0.1

Builds the data the website reads.

**Design**
* **Domestic games** are rated against a baseline per group of leagues (per role): LCK, LPL, LEC and LCS share one common baseline, LCP has its own.
* **International games** are rated against their own baseline (only international games).
* Every game is rated once and stored. The site's filters (league, split, regular season / playoffs, international) just choose which games get averaged.
* A player's rating for any selection of games:

  `rating = 1 + c(role) * sum(w * (game rating - 1)) / (sum(w) + K)`

  with `w = 1` for a domestic game and `w = INTL_WEIGHT` for an international game, `K` = small-sample shrinkage, and `c(role)` a fixed constant
  that gives player ratings a typical spread of about 0.15 within each role (so filters never change the scale).
* The rating is a **performance** rating relative to the games it is built from, not a measure of skill.

Needs `rating_core.py` and the match-data CSV in the same folder. Press **Run All**.

In [1]:
# ================= CONFIG =================
FILE = "2026_LoL_esports_match_data_from_OraclesElixir.csv"
BASELINE_GROUPS = {"main": ["LCK", "LPL", "LEC", "LCS"], "lcp": ["LCP"]}   # each group is rated against its OWN baseline
LEAGUES = [lg for grp in BASELINE_GROUPS.values() for lg in grp]            # leagues shown on the site
INTERNATIONAL = ["MSI", "WLDs", "FST", "EWC"]          # league codes of international events (check the list printed below)
K = 12                 # small-sample shrinkage (use the K_USE printed by the baseline-check notebook)
INTL_WEIGHT = 2.0      # one international game counts as this many domestic games in the overall rating
TARGET_SD = 0.15       # typical spread of player ratings inside a role
MIN_GAMES_CALIBRATION = 15
SPLIT_MAP = {}         # optional, e.g. {("LPL", "Split 1"): "Spring"} to give regions a common period name
OUT_DIR = "../docs/data"

In [2]:
import os, json
from datetime import datetime
import numpy as np
import pandas as pd
import rating_core as rc

df = pd.read_csv(FILE, low_memory=False)
cfg = dict(rc.CFG)

counts = df[df["position"] != "team"].groupby("league")["gameid"].nunique().sort_values(ascending=False)
print("All league codes in the file and their game counts:")
print(counts.to_string(), "\n")

def pool(leagues, label):
    parts = []
    for lg in leagues:
        p = rc.load_games(df, lg, include_playoffs=True)      # keep playoffs, they are tagged and filterable
        n = p["gameid"].nunique()
        if n < 10:
            print(f"{label}: {lg} has only {n} complete games, skipped"); continue
        parts.append(rc.add_raw_stats(p))
        print(f"{label}: {lg} {n} games")
    return pd.concat(parts, ignore_index=True) if parts else None

parts = []
for name, lgs in BASELINE_GROUPS.items():
    raw = pool(lgs, name)
    if raw is None or raw["gameid"].nunique() < 30:
        print(f"{name}: not enough games for its own baseline, skipped")
        continue
    rated = rc.rate(raw, cfg)                                 # this group's own baseline
    print(f"{name}: baseline built from {rated['gameid'].nunique()} games; stats skipped: {rated.attrs.get('skipped_stats')}")
    parts.append(rated)
dom = pd.concat(parts, ignore_index=True)
intl = pool(INTERNATIONAL, "international")
if intl is not None and intl["gameid"].nunique() >= 30:
    intl = rc.rate(intl, cfg)                                 # separate international baseline
    print("stats skipped in the international baseline:", intl.attrs.get("skipped_stats"))
else:
    print("not enough international games, international ratings will be empty")
    intl = None

print("\nSplit labels per league (domestic), with playoffs flag:")
print(dom.groupby(["league", "split", "playoffs"])["gameid"].nunique().to_string())

All league codes in the file and their game counts:
league
LPL          674
LAS          541
EM           524
LCK          502
LCKC         491
LEC          383
LJL          363
AL           350
PRM          322
CBLOL        299
LCP          283
LCS          283
NACL         281
LIT          280
LFL          275
ROL          261
CD           258
HLL          235
EWC          232
TCL          219
EBL          213
RL           211
NLC          210
LES          208
HC           201
LRS          187
LRN          176
HM           174
LPLOL        162
VCS          155
NL           149
HW           103
PCS           86
WSCI          85
MSI           71
DCGI          71
AM            63
WLDs          63
KeSPA Cup     61
FST           45
CCWS          40
AC            19
WLRQ           6
DCup           3 

main: LCK 502 games
main: LPL 548 games
main: LEC 383 games
main: LCS 283 games


c:\Ratings\lol-ratings\pipeline\rating_core.py:119: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  p["lg_" + name] = blend(p[cols], [1.0] * len(cols)) if cols else np.nan
c:\Ratings\lol-ratings\pipeline\rating_core.py:120: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  p["late"] = np.nan
c:\Ratings\lol-ratings\pipeline\rating_core.py:128: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.co

main: baseline built from 1716 games; stats skipped: []
lcp: LCP 283 games
lcp: baseline built from 283 games; stats skipped: []
international: MSI 71 games
international: WLDs 63 games
international: FST 45 games
international: EWC 232 games
stats skipped in the international baseline: []

Split labels per league (domestic), with playoffs flag:
league  split       playoffs
LCK     Cup         0           125
        Rounds 1-2  0           204
                    1            20
        Rounds 3-4  0            98
                    1            55
LCP     Split 1     0            70
                    1            32
        Split 2     0            69
                    1            32
        Split 3     0            80
LCS     Lock-In     0            59
        Spring      0            69
                    1            29
        Summer      0            64
                    1            39
LEC     Spring      0           111
                    1            30
        Sum

c:\Ratings\lol-ratings\pipeline\rating_core.py:119: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  p["lg_" + name] = blend(p[cols], [1.0] * len(cols)) if cols else np.nan
c:\Ratings\lol-ratings\pipeline\rating_core.py:120: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  p["late"] = np.nan
c:\Ratings\lol-ratings\pipeline\rating_core.py:128: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.co

In [3]:
# ---------- one table with every rated game ----------
def tidy(p, kind):
    out = pd.DataFrame({
        "player": p["playername"], "role": p["position"], "team": p["teamname"], "gameid": p["gameid"],
        "date": p["date"], "league": p["league"], "kind": kind,
        "stage": np.where(p["playoffs"] == 1, "playoffs", "regular"),
        "split": p["split"],
        "opponent": p["opponent"], "champion": p["champion"], "result": p["result"],
        "minutes": p["minutes"].round(1), "kills": p["kills"], "deaths": p["deaths"], "assists": p["assists"],
        "gd15": p["golddiffat15"],
        "r_early": p["r_early"], "r_late": p["r_late"], "r_overall": p["r_overall"],
    })
    out["period"] = [SPLIT_MAP.get((lg, sp), sp) for lg, sp in zip(out["league"], out["split"])]
    out["year"] = out["date"].dt.year
    return out

games = pd.concat([tidy(dom, "domestic")] + ([tidy(intl, "international")] if intl is not None else []), ignore_index=True)
games = games.sort_values("date").reset_index(drop=True)
print("rated player-games:", len(games), "| domestic:", int((games["kind"] == "domestic").sum()),
      "| international:", int((games["kind"] == "international").sum()))

# ---------- the rating for ANY selection of games ----------
C_BY_ROLE = {r: 1.0 for r in ["top", "jng", "mid", "bot", "sup"]}

def rate_players(g, K=K, W=INTL_WEIGHT, c_by_role=None, z=1.645):
    """Rating (with ~90% interval) per player for the games in g. Same formula the website uses."""
    c_by_role = c_by_role or C_BY_ROLE
    g = g.copy()
    g["w"] = np.where(g["kind"] == "international", W, 1.0)
    out = None
    for m in ["overall", "early", "late"]:
        r = g["r_" + m]
        wm = g["w"].where(r.notna(), 0.0)
        tmp = pd.DataFrame({"player": g["player"], "role": g["role"], "wm": wm,
                            "wdev": wm * (r.fillna(1.0) - 1), "w2": wm ** 2, "r": r})
        a = tmp.groupby(["player", "role"]).agg(n_eff=("wm", "sum"), s=("wdev", "sum"), w2=("w2", "sum"), sd=("r", "std"))
        c = a.index.get_level_values("role").map(c_by_role).to_numpy(dtype=float)
        den = a["n_eff"] + K
        a[m] = 1 + c * a["s"] / den
        half = z * c * a["sd"].fillna(0) * np.sqrt(a["w2"]) / den
        a[m + "_low"], a[m + "_high"] = a[m] - half, a[m] + half
        keep = [m, m + "_low", m + "_high"] + (["n_eff"] if m == "overall" else [])
        out = a[keep] if out is None else out.join(a[keep])
    info = g.groupby(["player", "role"]).agg(games=("gameid", "nunique"), team=("team", "last"))
    return out.join(info).reset_index()

# fixed spread constants: calibrated once, on the first baseline group (the main leagues), and used for every game
CAL_LEAGUES = list(BASELINE_GROUPS.values())[0]
base = rate_players(games[(games["kind"] == "domestic") & games["league"].isin(CAL_LEAGUES)], c_by_role={r: 1.0 for r in C_BY_ROLE})
q = base[base["games"] >= MIN_GAMES_CALIBRATION].copy()
q["dev"] = q["overall"] - 1
C_BY_ROLE = {r: float(TARGET_SD / q.loc[q["role"] == r, "dev"].std()) for r in C_BY_ROLE}
print("spread constants c(role):", {r: round(v, 2) for r, v in C_BY_ROLE.items()})

rated player-games: 24100 | domestic: 19990 | international: 4110
spread constants c(role): {'top': 2.89, 'jng': 2.47, 'mid': 2.45, 'bot': 2.69, 'sup': 2.08}


In [4]:
# ---------- default tables: domestic only, international only, overall ----------
dom_t = rate_players(games[games["kind"] == "domestic"]).add_prefix("dom_").rename(columns={"dom_player": "player", "dom_role": "role"})
all_t = rate_players(games)
if (games["kind"] == "international").any():
    int_t = rate_players(games[games["kind"] == "international"]).add_prefix("intl_").rename(columns={"intl_player": "player", "intl_role": "role"})
else:
    int_t = pd.DataFrame(columns=["player", "role"])
table = all_t.merge(dom_t[["player", "role", "dom_overall", "dom_games"]], on=["player", "role"], how="left") \
             .merge(int_t[["player", "role"] + [c for c in int_t.columns if c in ("intl_overall", "intl_overall_low", "intl_overall_high", "intl_games")]],
                    on=["player", "role"], how="left")
table["league"] = table["team"].map(games.groupby("team")["league"].agg(lambda s: s[s.isin(LEAGUES)].mode().iloc[0] if s.isin(LEAGUES).any() else s.iloc[0]))

show = ["league", "team", "player", "games", "dom_overall", "intl_overall", "overall", "overall_low", "overall_high", "early", "late"]
for role in ["top", "jng", "mid", "bot", "sup"]:
    x = table[(table["role"] == role) & (table["games"] >= MIN_GAMES_CALIBRATION)].sort_values("overall", ascending=False).head(10)
    print(role.upper(), "(top 10, all games)")
    print(x[[c for c in show if c in x.columns]].round(2).to_string(index=False), "\n")

# example of a filter: one league only
ex = rate_players(games[games["league"] == LEAGUES[0]])
print(f"Example filter, {LEAGUES[0]} only, best 5 overall:")
print(ex[ex["games"] >= MIN_GAMES_CALIBRATION].sort_values("overall", ascending=False).head(5)[["player", "role", "team", "games", "overall", "overall_low", "overall_high"]].round(2).to_string(index=False))

TOP (top 10, all games)
league                team    player  games  dom_overall  intl_overall  overall  overall_low  overall_high  early  late
   LCK               Gen.G      Kiin    124         1.33          1.42     1.38         1.28          1.49   1.23  1.39
   EWC                 LØS      Zest     23          NaN          1.36     1.36         1.14          1.57   1.10  1.46
   LPL     Bilibili Gaming       Bin    133         1.24          1.22     1.25         1.15          1.35   1.19  1.23
   LCK Hanwha Life Esports      Zeus    138         1.19          1.26     1.23         1.13          1.34   1.17  1.22
   LEC       Team Vitality Naak Nako     86         1.27          1.00     1.23         1.11          1.34   1.27  1.13
   LEC            Shifters   Rooster     61         1.18          1.21     1.22         1.10          1.34   1.25  1.13
   LPL   Ninjas in Pyjamas      HOYA     75         1.21           NaN     1.21         1.10          1.33   1.11  1.23
   LEC        Ka

In [5]:
# ---------- export ----------
os.makedirs(OUT_DIR, exist_ok=True)

fields = ["player", "role", "team", "date", "league", "kind", "stage", "period", "opponent", "champion",
          "result", "minutes", "kills", "deaths", "assists", "gd15", "r_early", "r_late", "r_overall"]
g_out = games[fields].copy()
g_out["date"] = g_out["date"].dt.strftime("%Y-%m-%d")
for c in ["r_early", "r_late", "r_overall"]:
    g_out[c] = g_out[c].round(3)
g_out["gd15"] = g_out["gd15"].round(0)
g_out.to_csv(os.path.join(OUT_DIR, "games_rated.csv"), index=False)
table.round(3).to_csv(os.path.join(OUT_DIR, "players_default_view.csv"), index=False)

def rows_of(frame):
    f = frame.astype(object).where(frame.notna(), None)
    return f.values.tolist()

payload = {
    "version": "0.1",
    "generated": datetime.now().isoformat(timespec="seconds"),
    "note": "Performance rating relative to the games it is built from. Not a measure of skill or of league strength.",
    "config": {"leagues": LEAGUES, "baseline_groups": BASELINE_GROUPS, "international": INTERNATIONAL, "K": K, "intl_weight": INTL_WEIGHT, "min_games": MIN_GAMES_CALIBRATION,
               "target_sd": TARGET_SD, "c_by_role": C_BY_ROLE,
               "baselines": "domestic games: one baseline per group of leagues (see baseline_groups); international games: their own baseline",
               "formula": "rating = 1 + c(role) * sum(w * (game_rating - 1)) / (sum(w) + K)",
               "early_weights": cfg["EARLY_WEIGHTS"], "late_groups": cfg["LATE_GROUPS"],
               "late_weights": cfg["LATE_WEIGHTS"], "early_share": cfg["EARLY_SHARE"]},
    "game_fields": fields,
    "games": rows_of(g_out),
    "reference_default_view": json.loads(table.round(3).astype(object).where(table.notna(), None).to_json(orient="records")),
}
path = os.path.join(OUT_DIR, "site_data.json")
with open(path, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, default=str, separators=(",", ":"))
for name in sorted(os.listdir(OUT_DIR)):
    print(f"{name:26s} {os.path.getsize(os.path.join(OUT_DIR, name)) / 1024:8.0f} KB")
print("Saved in:", os.path.abspath(OUT_DIR))

games_rated.csv                2839 KB
players_default_view.csv         45 KB
site_data.json                 3489 KB
Saved in: c:\Ratings\lol-ratings\docs\data
